# Первый парсер Excel-отчета в pandas DataFrame

Цель: превратить отчет `Продажи по контрагентам (по оплате)` из иерархического Excel-вида в аналитическую таблицу, где **одна строка = один платеж / одно поступление на расчетный счет**.

Главное правило: наружу отдаем только `payment_date`. Внутри парсера дата сначала берется из колонки Excel, но если дата в тексте документа распарсилась и отличается от колонки, итоговой `payment_date` считается дата из документа.

In [1]:
from pathlib import Path
import importlib.util
import re
import sys

import pandas as pd

# Старые .xls файлы pandas читает через пакет xlrd.
# Если в ноутбуке будет ошибка про xlrd, установи его один раз:
# %pip install xlrd
#
# В текущей рабочей сессии xlrd был поставлен во временную папку, поэтому добавим ее,
# если обычный import его не видит.
if importlib.util.find_spec("xlrd") is None:
    temp_xlrd = Path("/private/tmp/codex-xlrd")
    if temp_xlrd.exists():
        sys.path.insert(0, str(temp_xlrd))

if importlib.util.find_spec("xlrd") is None:
    raise ModuleNotFoundError("Нужен пакет xlrd: выполни в отдельной ячейке `%pip install xlrd`")

## Настройки исходного файла

In [2]:
SOURCE_FILE = Path("Продажи по контрагентам (по оплате)/Продажи по контрагентам (по оплате) за Апрель 2026 г..xls")
PERIOD_MONTH = pd.Timestamp("2026-04-01")

SOURCE_FILE.exists(), SOURCE_FILE

(True,
 PosixPath('Продажи по контрагентам (по оплате)/Продажи по контрагентам (по оплате) за Апрель 2026 г..xls'))

## Вспомогательные функции

`parse_payment_report` делает основную работу:

- читает лист без заголовков;
- берет даты оплат из колонок отчета;
- идет по блокам `контрагент -> договор -> документы`;
- для каждой суммы в дневной колонке создает отдельную строку таблицы.

In [3]:
def _clean_text(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return re.sub(r"\s+", " ", text) or None


def _parse_report_date(value):
    """Parse Excel column header like 01.04.26 into Timestamp."""
    text = _clean_text(value)
    if not text:
        return None
    return pd.to_datetime(text, format="%d.%m.%y")


def _parse_ru_date(value):
    """Parse text date like 31.12.2024 into Timestamp."""
    if not value:
        return pd.NaT
    return pd.to_datetime(value, format="%d.%m.%Y", errors="coerce")


def parse_contract(contract_raw):
    contract_raw = _clean_text(contract_raw)
    if not contract_raw:
        return pd.Series({"contract_number": None, "contract_date": pd.NaT})

    match = re.search(r"\s+от\s+(\d{2}\.\d{2}\.\d{4})$", contract_raw)
    if not match:
        return pd.Series({"contract_number": contract_raw, "contract_date": pd.NaT})

    return pd.Series({
        "contract_number": contract_raw[:match.start()].strip(),
        "contract_date": _parse_ru_date(match.group(1)),
    })


def parse_payment_doc(payment_doc_raw):
    payment_doc_raw = _clean_text(payment_doc_raw)
    if not payment_doc_raw:
        return pd.Series({"payment_doc_number": None, "payment_doc_date": pd.NaT})

    match = re.search(r"№\s*([^\s]+)\s+от\s+(\d{2}\.\d{2}\.\d{4})", payment_doc_raw)
    if not match:
        return pd.Series({"payment_doc_number": None, "payment_doc_date": pd.NaT})

    return pd.Series({
        "payment_doc_number": match.group(1),
        "payment_doc_date": _parse_ru_date(match.group(2)),
    })


def parse_counterparty(counterparty_raw):
    counterparty_raw = _clean_text(counterparty_raw)
    result = {
        "legal_entity": None,
        "brand": None,
        "store_location_raw": None,
        "city_or_area": None,
    }
    if not counterparty_raw:
        return pd.Series(result)

    if "Магазин Светофор" in counterparty_raw:
        left, location = counterparty_raw.split("Магазин Светофор", 1)
        result["legal_entity"] = left.strip()
        result["brand"] = "Светофор"
        result["store_location_raw"] = location.strip(" ,") or None
    elif "Магазин Маяк" in counterparty_raw:
        left, location = counterparty_raw.split("Магазин Маяк", 1)
        result["legal_entity"] = left.strip().strip(",")
        result["brand"] = "Маяк"
        result["store_location_raw"] = location.strip(" ,") or None
    else:
        result["legal_entity"] = counterparty_raw

    location = result["store_location_raw"] or ""
    city_match = re.search(r"((?:г\.|рп\.|пгт|с\.)\s*[^,]+)", location)
    if city_match:
        result["city_or_area"] = city_match.group(1).strip()

    return pd.Series(result)


def parse_payment_report(path, period_month=None, sheet_name=0):
    path = Path(path)
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None, engine="xlrd")

    headers = raw.iloc[0].tolist()
    date_columns = {
        col_idx: _parse_report_date(headers[col_idx])
        for col_idx in range(2, len(headers))
        if _clean_text(headers[col_idx])
    }

    total_rows = raw[raw[0].astype(str).str.strip().eq("Итого")]
    report_total = float(total_rows.iloc[0, 1]) if len(total_rows) else None

    rows = []
    i = 3
    while i < len(raw):
        first_cell = _clean_text(raw.iat[i, 0])
        if not first_cell or first_cell == "Итого" or first_cell.startswith("*"):
            break

        # Ожидаемый блок: строка контрагента, затем строка договора, затем строки документов.
        counterparty_raw = first_cell
        counterparty_month_total = float(raw.iat[i, 1]) if pd.notna(raw.iat[i, 1]) else None

        contract_raw = _clean_text(raw.iat[i + 1, 0]) if i + 1 < len(raw) else None

        j = i + 2
        while j < len(raw):
            payment_doc_raw = _clean_text(raw.iat[j, 0])
            if not payment_doc_raw or not payment_doc_raw.startswith("Поступление на расчетный счет"):
                break

            doc_info = parse_payment_doc(payment_doc_raw)
            doc_number = doc_info["payment_doc_number"]
            doc_date = doc_info["payment_doc_date"]

            # В норме у документа одна сумма в одной дневной колонке.
            # Если вдруг будет несколько ненулевых дневных колонок, создадим несколько строк.
            for col_idx, payment_column_date in date_columns.items():
                amount = raw.iat[j, col_idx]
                if pd.isna(amount) or float(amount) == 0:
                    continue

                payment_date = doc_date if pd.notna(doc_date) and payment_column_date != doc_date else payment_column_date

                rows.append({
                    "source_file": path.name,
                    "period_month": period_month,
                    "counterparty_raw": counterparty_raw,
                    "contract_raw": contract_raw,
                    "payment_doc_raw": payment_doc_raw,
                    "payment_doc_number": doc_number,
                    "payment_date": payment_date,
                    "amount": float(amount),
                    "counterparty_month_total": counterparty_month_total,
                    "report_total": report_total,
                    "source_row_number": j + 1,
                    "source_column": headers[col_idx],
                })

            j += 1

        i = j

    payments = pd.DataFrame(rows)
    if payments.empty:
        return payments

    payments = pd.concat([
        payments,
        payments["counterparty_raw"].apply(parse_counterparty),
        payments["contract_raw"].apply(parse_contract),
    ], axis=1)

    ordered_columns = [
        "source_file",
        "period_month",
        "counterparty_raw",
        "legal_entity",
        "brand",
        "store_location_raw",
        "city_or_area",
        "contract_raw",
        "contract_number",
        "contract_date",
        "payment_doc_raw",
        "payment_doc_number",
        "payment_date",
        "amount",
        "counterparty_month_total",
        "report_total",
        "source_row_number",
        "source_column",
    ]
    return payments[ordered_columns].sort_values(["payment_date", "counterparty_raw", "payment_doc_number"]).reset_index(drop=True)

## Запуск парсера

In [4]:
payments_df = parse_payment_report(SOURCE_FILE, period_month=PERIOD_MONTH)

payments_df.shape

(347, 18)

In [5]:
payments_df.head(10)

,source_file,period_month,counterparty_raw,legal_entity,brand,store_location_raw,city_or_area,contract_raw,contract_number,contract_date,payment_doc_raw,payment_doc_number,payment_date,amount,counterparty_month_total,report_total,source_row_number,source_column
0,Продажи по контрагентам (по оплате) за Апрель ...,2026-04-01,"ООО ""Восторг 76"" Магазин Маяк, г.Серпухов, Бор...","ООО ""Восторг 76""",Маяк,"г.Серпухов, Борисово д, Квартал А тер., 47,1",г.Серпухов,36/ВС от 20.09.2023,36/ВС,2023-09-20,Поступление на расчетный счет № 1589 от 01.04....,1589,2026-04-01,12316.8,24114.4,3641845.57,334,01.04.26
1,Продажи по контрагентам (по оплате) за Апрель ...,2026-04-01,"ООО""Восторг 76"" Магазин Маяк, г. Калуга, ул. М...","ООО""Восторг 76""",Маяк,"г. Калуга, ул. Московская, д.290, строение 8.",г. Калуга,36/ВС от 20.09.2023,36/ВС,2023-09-20,Поступление на расчетный счет № 2603 от 01.04....,2603,2026-04-01,1508.0,20800.0,3641845.57,409,01.04.26
2,Продажи по контрагентам (по оплате) за Апрель ...,2026-04-01,"Торгсервис 150 ООО Магазин Светофор МО, г. Куб...",Торгсервис 150 ООО,Светофор,"МО, г. Кубинка, ул. Железнодорожная, д. 1А",г. Кубинка,ТС 150/25-241 ПМСК от 31.12.2024,ТС 150/25-241 ПМСК,2024-12-31,Поступление на расчетный счет № 772 от 01.04.2026,772,2026-04-01,6180.0,8755.0,3641845.57,559,01.04.26
3,Продажи по контрагентам (по оплате) за Апрель ...,2026-04-01,Торгсервис 150 ООО Магазин Светофор Московская...,Торгсервис 150 ООО,Светофор,"Московская обл. рп. Серебряные Пруды, ул. Прив...",рп. Серебряные Пруды,ТС150/25-241 ПМСК от 31.12.2024,ТС150/25-241 ПМСК,2024-12-31,Поступление на расчетный счет № 695 от 01.04.2026,695,2026-04-01,26591.5,83424.2,3641845.57,7,01.04.26
4,Продажи по контрагентам (по оплате) за Апрель ...,2026-04-01,Торгсервис 150 ООО Магазин Светофор г. Воскрес...,Торгсервис 150 ООО,Светофор,"г. Воскресенск, ул. 2-я Заводская д.16, стр.2",г. Воскресенск,ТС150/25-241 ПМСК от 31.12.2024,ТС150/25-241 ПМСК,2024-12-31,Поступление на расчетный счет № 992 от 01.04.2026,992,2026-04-01,15591.2,62827.4,3641845.57,29,01.04.26
5,Продажи по контрагентам (по оплате) за Апрель ...,2026-04-01,"Торгсервис 150 ООО Магазин Светофор г. Москва,...",Торгсервис 150 ООО,Светофор,"г. Москва, Огородный пр-зд, д. 10, стр.4",г. Москва,ТС150/25-241 ПМСК от 31.12.2024,ТС150/25-241 ПМСК,2024-12-31,Поступление на расчетный счет № 831 от 01.04.2026,831,2026-04-01,19570.0,34247.5,3641845.57,181,01.04.26
6,Продажи по контрагентам (по оплате) за Апрель ...,2026-04-01,Торгсервис 150 ООО Магазин Светофор г. Щёлково...,Торгсервис 150 ООО,Светофор,"г. Щёлково, ул. Радужная, стр.1А",г. Щёлково,ТС 150/25-241 ПМСК от 31.12.2024,ТС 150/25-241 ПМСК,2024-12-31,Поступление на расчетный счет № 794 от 01.04.2026,794,2026-04-01,9012.5,15192.5,3641845.57,463,01.04.26
7,Продажи по контрагентам (по оплате) за Апрель ...,2026-04-01,Торгсервис 50 ООО Магазин Светофор Павлово Пос...,Торгсервис 50 ООО,Светофор,"Павлово Посадский р-н, д. Кузнецы, д. 76а",None,ТС50/25-241 ПМСК от 31.12.2024,ТС50/25-241 ПМСК,2024-12-31,Поступление на расчетный счет № 739 от 01.04.2026,739,2026-04-01,18692.7,37669.1,3641845.57,147,01.04.26
8,Продажи по контрагентам (по оплате) за Апрель ...,2026-04-01,"Торгсервис 50 ООО Магазин Светофор г. Клин, д....",Торгсервис 50 ООО,Светофор,"г. Клин, д.Першутино, д.1А, стр.1",г. Клин,ТС-50/24-008МО2 от 09.02.2024,ТС-50/24-008МО2,2024-02-09,Поступление на расчетный счет № 944 от 01.04.2026,944,2026-04-01,7210.0,23690.0,3641845.57,348,01.04.26
9,Продажи по контрагентам (по оплате) за Апрель ...,2026-04-01,"Торгсервис 50 ООО Магазин Светофор г. Клин, ул...",Торгсервис 50 ООО,Светофор,"г. Клин, ул. Победы, влд.11, пом.3",г. Клин,ТС 50/25-241 ПМСК от 31.12.2024,ТС 50/25-241 ПМСК,2024-12-31,Поступление на расчетный счет № 1346 от 01.04....,1346,2026-04-01,6388.3,47588.3,3641845.57,82,01.04.26


In [14]:
output_path = Path("payments_df.xlsx")

payments_df.to_excel(
    output_path,
    index=False
)

output_path


PosixPath('payments_df.xlsx')

## Проверки качества парсинга

In [6]:
parsed_total = payments_df["amount"].sum()
report_total = payments_df["report_total"].dropna().iloc[0]

print(f"Сумма платежей из распарсенной таблицы: {parsed_total:,.2f}")
print(f"Итог из Excel-отчета:                 {report_total:,.2f}")
print(f"Разница:                              {parsed_total - report_total:,.2f}")

Сумма платежей из распарсенной таблицы: 3,641,845.57
Итог из Excel-отчета:                 3,641,845.57
Разница:                              0.00


In [7]:
# Внутренняя проверка расхождений дат: наружу в payments_df не добавляем служебные колонки.
# Здесь временно достаем дату документа из payment_doc_raw и сравниваем ее с итоговой payment_date.
audit_dates = payments_df[["payment_doc_raw", "payment_date", "amount"]].copy()
audit_dates = pd.concat([audit_dates, audit_dates["payment_doc_raw"].apply(parse_payment_doc)], axis=1)

# После правила выбора даты расхождений быть не должно: если дата документа была другой,
# payment_date уже должна быть равна дате документа.
date_mismatch = audit_dates[
    audit_dates["payment_doc_date"].notna()
    & (audit_dates["payment_date"] != audit_dates["payment_doc_date"])
]

date_mismatch.head(20)

,payment_doc_raw,payment_date,amount,payment_doc_number,payment_doc_date


In [8]:
# Если парсер сработал правильно, после применения правила выбора даты список пустой.
assert date_mismatch.empty

date_mismatch.shape[0]


0

## Примеры анализа в pandas

In [9]:
payments_by_day = (
    payments_df
    .groupby("payment_date", as_index=False)["amount"]
    .sum()
    .sort_values("payment_date")
)

payments_by_day

,payment_date,amount
0,2026-04-01,220760.28
1,2026-04-03,135913.17
2,2026-04-06,597735.20
3,2026-04-08,324889.92
4,2026-04-10,161672.10
5,2026-04-13,419874.80
6,2026-04-15,75192.40
7,2026-04-17,163590.00
8,2026-04-20,465115.30
9,2026-04-22,129453.60


In [10]:
payments_by_group = (
    payments_df
    .groupby(["legal_entity", "brand"], dropna=False, as_index=False)
    .agg(
        amount=("amount", "sum"),
        payments_count=("payment_doc_number", "count"),
        counterparties_count=("counterparty_raw", "nunique"),
    )
    .sort_values("amount", ascending=False)
)

payments_by_group

,legal_entity,brand,amount,payments_count,counterparties_count
4,Торгсервис 150 ООО,Светофор,1514894.20,114,55
8,Торгсервис 50 ООО,Светофор,1198893.40,110,43
10,Торгсервис 71 ООО,Светофор,526899.70,64,27
0,Восторг 76 ООО,Маяк,187898.57,34,13
2,"ООО ""Восторг 76""",Маяк,74201.80,7,2
3,"ООО""Восторг 76""",Маяк,42780.40,7,2
6,"Торгсервис 150 ООО магазин Светофор, г. Люберц...",NaN,41972.50,3,1
7,Торгсервис 50,Светофор,29539.00,3,1
1,Восторг 76 ООО,Светофор,11587.50,1,1
9,Торгсервис 50 ООО Магазин светофор г. Волокола...,NaN,7369.10,2,1


In [11]:
top_counterparties = (
    payments_df
    .groupby(["counterparty_raw", "legal_entity", "brand"], dropna=False, as_index=False)
    .agg(
        amount=("amount", "sum"),
        payments_count=("payment_doc_number", "count"),
        first_payment_date=("payment_date", "min"),
        last_payment_date=("payment_date", "max"),
    )
    .sort_values("amount", ascending=False)
    .head(20)
)

top_counterparties

,counterparty_raw,legal_entity,brand,amount,payments_count,first_payment_date,last_payment_date
21,Торгсервис 150 ООО Магазин Светофор Московская...,Торгсервис 150 ООО,Светофор,83424.2,4,2026-04-01,2026-04-27
52,Торгсервис 150 ООО Магазин Светофор г. Подольс...,Торгсервис 150 ООО,Светофор,78022.5,2,2026-04-08,2026-04-27
104,Торгсервис 50 ООО Магазин Светофор г. Сергиев-...,Торгсервис 50 ООО,Светофор,71524.9,4,2026-04-06,2026-04-27
108,"Торгсервис 50 ООО Магазин Светофор г. Ступино,...",Торгсервис 50 ООО,Светофор,65377.5,4,2026-04-06,2026-04-29
24,Торгсервис 150 ООО Магазин Светофор г. Воскрес...,Торгсервис 150 ООО,Светофор,62827.4,4,2026-04-01,2026-04-29
112,Торгсервис 50 ООО Магазин Светофор г.Орехово-З...,Торгсервис 50 ООО,Светофор,61826.8,3,2026-04-06,2026-04-27
37,Торгсервис 150 ООО Магазин Светофор г. Можайск...,Торгсервис 150 ООО,Светофор,58402.0,3,2026-04-06,2026-04-27
58,"Торгсервис 150 ООО Магазин Светофор г. Чехов, ...",Торгсервис 150 ООО,Светофор,56661.5,3,2026-04-08,2026-04-27
100,Торгсервис 50 ООО Магазин Светофор г. Раменско...,Торгсервис 50 ООО,Светофор,56392.5,3,2026-04-06,2026-04-27
28,Торгсервис 150 ООО Магазин Светофор г. Домодед...,Торгсервис 150 ООО,Светофор,54075.0,2,2026-04-08,2026-04-24


## Пример SQL поверх pandas

Без отдельной базы можно временно загрузить DataFrame в SQLite и делать SQL-запросы.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
payments_df.to_sql("payments", conn, index=False, if_exists="replace")

pd.read_sql_query(
    """
    select
        payment_date,
        round(sum(amount), 2) as amount,
        count(*) as payments_count
    from payments
    group by payment_date
    order by payment_date
    """,
    conn,
)

In [ ]:
pd.read_sql_query(
    """
    select
        legal_entity,
        brand,
        round(sum(amount), 2) as amount,
        count(*) as payments_count,
        count(distinct counterparty_raw) as counterparties_count
    from payments
    group by legal_entity, brand
    order by amount desc
    """,
    conn,
)

## Экспорт результата

После проверки можно сохранить нормализованную таблицу в CSV или Parquet.

In [ ]:
OUTPUT_CSV = Path("payments_april_2026_first_try.csv")
payments_df.to_csv(OUTPUT_CSV, index=False)

OUTPUT_CSV